In [ ]:
import sys
sys.path.insert(1, '/home/b/b309076/Python/Utils')


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import cm
from scipy.interpolate import interpn
from netCDF4 import Dataset
import netCDF4 as nc
import Utils as ut
import math
from calendar import monthrange
import matplotlib.colors as colors
#import pyshtools as pysh
from numpy.fft import fft, ifft, fft2, ifft2
import xarray as xr
import robfft_tape as robfft
import pylops

In [ ]:
# script to calculate spectral power and momentum fluxes (u'w') as function of zonal phase speed from ICON 6-hourly output data

# version 1 (7.10.2025):
# now includes: filtering of fields for background (removing scales larger than 2000 km), tapering of fields at the edges of boxes;
# using the actual latitude for calculation of coriolis parameter f
# multiplying the power and momentum flux by factor 2 after masking, since half the power is assigned to opposite quandrant in (kx,ky) space. 
#   [with factor 2, the summed momentum flux equals the directly calculated momentum flux in the box].
## version 1 (12.1.2026): includes horizontal momentum flux u'v'


#ihour0 = 1 # #timesetp in file
day0 = 24  #25 
day1 = 24
ndays =  day1-day0+1 # + day0
month = 8

dx = 2000 # km , size of Box
latCenters = (-70,-60,-50,-40,-30,-20,-10,0)

dlon = dx/(6370*np.cos(latCenters[1]*np.pi/180)*np.pi/180) #use dlon from central lat0 to allow for same number of boxes for each lat circle
print(dlon)
lonCenters = np.arange(-180+dlon/2,180-dlon/2,dlon/2)
print(lonCenters)

#lat0 = -60 #-45 # center latitude
#lon0 = -150 #-67 # center longitude

Path = '/work/bd1022/b309076/ICON/'

file_name_tuvw_ZM = f"{Path}/run_ICON_R2B7_ssoD_GC20/output/ICON_DOM01_HL_20150825T120000Z.nc"
tuvZ_ZM_data      = Dataset(file_name_tuvw_ZM,"r")
      # grid
lat   = tuvZ_ZM_data.variables["lat"][:]                    # latitudes
lon   = tuvZ_ZM_data.variables["lon"][:]
zlev  = tuvZ_ZM_data.variables["alt"][:]
tuvZ_ZM_data.close()
zlev = zlev*1e-3

dlat = dx/111 


#get power, uw, wl per phase speed
# define phase speed array to bin data on.
c_i = np.arange(-300,310,10)
c_central = (c_i[0:len(c_i)-1] +c_i[1:len(c_i)])*0.5

for iday in np.arange(day0,day1+1):
    day = iday
    print(month,day)
    file_name = f"{Path}/run_ICON_R2B7_ssoD_GC20/output/ICON_DOM01_HL_2015{ut.Fint2str(month,2)}{ut.Fint2str(day,2)}T120000Z.nc"
    data = xr.open_dataset(file_name, engine="netcdf4")
    print(data)

    #define arrays for goal variables
    power_per_c = np.zeros((4,len(latCenters),len(lonCenters),len(c_i)-1,len(zlev)-1,3))
    uw_per_c = np.zeros((4,len(latCenters),len(lonCenters),len(c_i)-1,len(zlev)-1,3))
    uv_per_c = np.zeros((4,len(latCenters),len(lonCenters),len(c_i)-1,len(zlev)-1,3))
    wl_per_c = np.zeros((4,len(latCenters),len(lonCenters),len(c_i)-1,len(zlev)-1,3))
    wlz_per_c = np.zeros((4,len(latCenters),len(lonCenters),len(c_i)-1,len(zlev)-1,3))
    Uwind_mean = np.zeros((4,len(latCenters),len(lonCenters),len(zlev)))
    Vwind_mean = np.zeros((4,len(latCenters),len(lonCenters),len(zlev)))

    for ihour in np.arange(0,4):

        ilat0 = 0
        for lat00 in latCenters:
            #lat boundaries
            lat1 = lat00 - dlat/2
            lat2 = lat00 + dlat/2
            ilon0 = 0

            #remove WN21 background at the given latitude range
            data_rbckg = robfft.remove_background(data.sel(lat=slice(lat1,lat2)),dim_lat='lat',dim_lon='lon',wl_cutoff=2000)

            for lon00 in lonCenters: 
                lon1 = lon00 - dlon/2
                lon2 = lon00 + dlon/2
                print('Box from Lat:',lat1, ' to ', lat2, ', Lon: ',lon1, ' to ', lon2)

                #select box
                d_sel_wbck = data.sel(time=data.time[ihour],lon=slice(lon1,lon2),lat=slice(lat1,lat2)).copy() #with background
                d_sel = data_rbckg.sel(time=data.time[ihour],lon=slice(lon1,lon2),lat=slice(lat1,lat2)).copy() #without background
                # no background, but add mean over box for calculation of Umean, Vmean, Tmean etc
                d_sel = d_sel + d_sel_wbck.mean(dim=['lat','lon'])

                # and change coordinates to be in meter
                coord_x1 = (d_sel.lon - lon00)*dx*1e3/dlon
                coord_y1 = (d_sel.lat - lat00)*dx*1e3/dlat
                d_sel = d_sel.assign_coords(lon = coord_x1)
                d_sel = d_sel.assign_coords(lat = coord_y1)

                # compute_vertical_wavenumber_and_intrinsic_frequency(ds,var,dim_x='x',dim_y='y',dim_z='z'):
                # !! assumes x and y in [meter], needs converted coordinates 
                X = robfft.compute_vertical_wavenumber_and_intrinsic_frequency(d_sel,'w',dim_x='lon',dim_y='lat',dim_z='alt',tapering= True,lat_mean=lat00)
                #print(X)
                #mom flux 
                UW  = robfft.compute_momentum_flux(d_sel,dim_x='lon',dim_y='lat',dim_z='alt',tapering= True)
                UV  = robfft.compute_momentum_flux_uv(d_sel,dim_x='lon',dim_y='lat',dim_z='alt',tapering= True)
                # energy flux
                PW = robfft.compute_energy_flux(d_sel,dim_x='lon',dim_y='lat',dim_z='alt',tapering= True)
                #background winds
                Uwind_mean[ihour,ilat0,ilon0,:]     = d_sel['u'].mean(dim=('lon','lat'))
                Vwind_mean[ihour,ilat0,ilon0,:]     = d_sel['v'].mean(dim=('lon','lat'))

        
                Ny = X.sizes['ky']
                Nx = X.sizes['kx']
                ly = 2*np.pi/X.ky #*(dx/Ny)
                lx = 2*np.pi/X.kx #*(dx/Nx)

                #array of horizontal wavelength
                wl_h = np.zeros((len(X.kx.values),len(X.ky.values)))
                lx.loc[dict(kx=0)] = dx*1e3
                ly.loc[dict(ky=0)] = dx*1e3
                for iky in np.arange(0,len(X.ky.values)):
                    wl_h[:,iky] = np.sqrt(lx**2 + ly[iky]**2)

                dkx = 1/len(X.kx.values) #(np.abs( X.kx.isel(kx=0)-X.kx.isel(kx=1))
                dky = 1/len(X.ky.values)  #np.abs( X.ky.isel(ky=0)-X.ky.isel(ky=1))

                for ilev in np.arange(0,len(X.alt.values)):
                    c_lev1 = X.omega_shifted.isel(alt=ilev)/X.kx
                    mask = PW.isel(alt=ilev)*X.m.isel(alt=ilev)  ## mask: only where p'w' and m are of opposite sign!
                    c_lev2 = c_lev1.where(mask<0,np.nan)
                    ## and only for kx,ky with at least X% of max. spectral power
                    power = X.spectral_power.isel(alt=ilev)*(dkx*dky)**2
                    powerTH = 0*power.max() # at least 10%  of max
                    c_lev = c_lev2.where(power>powerTH,np.nan)
                    wlz = 2*np.pi/X.m.isel(alt=ilev)
                    uwsel = UW.uw.isel(alt=ilev)*(dkx*dky)**2
                    uvsel = UV.uv.isel(alt=ilev)*(dkx*dky)**2
                    # all
                    power_per_c[ihour,ilat0,ilon0,:,ilev,0], c_edges = np.histogram(c_lev.pw.values.flatten(),bins=c_i,density=False,weights=power.values.flatten())
                    uw_per_c[ihour,ilat0,ilon0,:,ilev,0], c_edges = np.histogram(c_lev.pw.values.flatten(),bins=c_i,density=False,weights=uwsel.values.flatten())
                    uv_per_c[ihour,ilat0,ilon0,:,ilev,0], c_edges = np.histogram(c_lev.pw.values.flatten(),bins=c_i,density=False,weights=uvsel.values.flatten())
                    wl_per_c[ihour,ilat0,ilon0,:,ilev,0], c_edges = np.histogram(c_lev.pw.values.flatten(),bins=c_i,density=False,weights=wl_h.flatten()*power.values.flatten()) #wl, weighted by power    
                    wlz_per_c[ihour,ilat0,ilon0,:,ilev,0], c_edges = np.histogram(c_lev.pw.values.flatten(),bins=c_i,density=False,weights=wlz.values.flatten()*power.values.flatten()) #wl, weighted by power    
                    power_tot = power_per_c[ihour,ilat0,ilon0,:,ilev,0].copy()
                    wl_per_c[ihour,ilat0,ilon0,:,ilev,0] = wl_per_c[ihour,ilat0,ilon0,:,ilev,0]/power_tot #average horizontal wavelength in each bin,  weighted by power
                    wlz_per_c[ihour,ilat0,ilon0,:,ilev,0] = wlz_per_c[ihour,ilat0,ilon0,:,ilev,0]/power_tot #average horizontal wavelength in each bin,  weighted by power
                    #only upward    
                    c_lev_up = c_lev.where(X.m.isel(alt=ilev)<0)
                    power_per_c[ihour,ilat0,ilon0,:,ilev,1], c_edges = np.histogram(c_lev_up.pw.values.flatten(),bins=c_i,density=False,weights=power.values.flatten())
                    uw_per_c[ihour,ilat0,ilon0,:,ilev,1], c_edges = np.histogram(c_lev_up.pw.values.flatten(),bins=c_i,density=False,weights=uwsel.values.flatten())
                    uv_per_c[ihour,ilat0,ilon0,:,ilev,1], c_edges = np.histogram(c_lev_up.pw.values.flatten(),bins=c_i,density=False,weights=uvsel.values.flatten())
                    wl_per_c[ihour,ilat0,ilon0,:,ilev,1], c_edges = np.histogram(c_lev_up.pw.values.flatten(),bins=c_i,density=False,weights=wl_h.flatten()*power.values.flatten()) #wl, weighted by power
                    wlz_per_c[ihour,ilat0,ilon0,:,ilev,1], c_edges = np.histogram(c_lev_up.pw.values.flatten(),bins=c_i,density=False,weights=wlz.values.flatten()*power.values.flatten()) #wl, weighted by power
                    power_tot = power_per_c[ihour,ilat0,ilon0,:,ilev,1].copy()
                    wl_per_c[ihour,ilat0,ilon0,:,ilev,1] = wl_per_c[ihour,ilat0,ilon0,:,ilev,1]/power_tot #average horizontal wavelength in each bin,  weighted by power
                    wlz_per_c[ihour,ilat0,ilon0,:,ilev,1] = wlz_per_c[ihour,ilat0,ilon0,:,ilev,1]/power_tot #average horizontal wavelength in each bin,  weighted by power
                    # only downward
                    c_lev_do = c_lev.where(X.m.isel(alt=ilev)>0)
                    power_per_c[ihour,ilat0,ilon0,:,ilev,2], c_edges = np.histogram(c_lev_do.pw.values.flatten(),bins=c_i,density=False,weights=power.values.flatten())
                    uw_per_c[ihour,ilat0,ilon0,:,ilev,2], c_edges = np.histogram(c_lev_do.pw.values.flatten(),bins=c_i,density=False,weights=uwsel.values.flatten())
                    uv_per_c[ihour,ilat0,ilon0,:,ilev,2], c_edges = np.histogram(c_lev_do.pw.values.flatten(),bins=c_i,density=False,weights=uvsel.values.flatten())
                    wl_per_c[ihour,ilat0,ilon0,:,ilev,2], c_edges = np.histogram(c_lev_do.pw.values.flatten(),bins=c_i,density=False,weights=wl_h.flatten()*power.values.flatten()) #wl, weighted by power
                    wlz_per_c[ihour,ilat0,ilon0,:,ilev,2], c_edges = np.histogram(c_lev_do.pw.values.flatten(),bins=c_i,density=False,weights=wlz.values.flatten()*power.values.flatten()) #wl, weighted by power
                    power_tot = power_per_c[ihour,ilat0,ilon0,:,ilev,2].copy()
                    wl_per_c[ihour,ilat0,ilon0,:,ilev,2] = wl_per_c[ihour,ilat0,ilon0,:,ilev,2]/power_tot #average horizontal wavelength in each bin,  weighted by power
                    wlz_per_c[ihour,ilat0,ilon0,:,ilev,2] = wlz_per_c[ihour,ilat0,ilon0,:,ilev,2]/power_tot #average horizontal wavelength in each bin,  weighted by power



                ilon0 = ilon0 +1
            ilat0 = ilat0 + 1

    #factor 2 because of masking
    power_per_c = power_per_c*2
    uw_per_c = uw_per_c*2
    uv_per_c = uv_per_c*2



    #write to file for current day
    fn = f"{Path}/run_ICON_R2B7_ssoD_GC20/output/GWspec/ICON_DOM01_HL_2015{ut.Fint2str(month,2)}{ut.Fint2str(day,2)}T120000Z_GWspec0to70S.nc"
    ds = nc.Dataset(fn, 'w', format='NETCDF4')
    # dimensions
    dimtime = ds.createDimension('time', None)
    dimlat = ds.createDimension('lat0', len(latCenters))
    dimlon = ds.createDimension('lon0', len(lonCenters))
    dimci = ds.createDimension('c_i', len(c_central))
    dimlev = ds.createDimension('lev', len(zlev)-1)
    dimlev1 = ds.createDimension('lev1', len(zlev))
    dimcase = ds.createDimension('case', 3)
    # create variables
    Vlat0 = ds.createVariable('lat0', 'f4', ('lat0',))
    Vlon0 = ds.createVariable('lon0', 'f4', ('lon0',))
    Vlev0 = ds.createVariable('lev', 'f4', ('lev',))
    Vlev1 = ds.createVariable('lev1', 'f4', ('lev1',))
    Vci = ds.createVariable('c_i', 'f4', ('c_i',))

    Vpower_per_c = ds.createVariable('power_per_c', 'f4', ('time', 'lat0','lon0','c_i','lev','case'))
    Vuw_per_c = ds.createVariable('uw_per_c', 'f4', ('time', 'lat0','lon0','c_i','lev','case'))
    Vuv_per_c = ds.createVariable('uv_per_c', 'f4', ('time', 'lat0','lon0','c_i','lev','case'))
    Vwl_per_c = ds.createVariable('wl_per_c', 'f4', ('time', 'lat0','lon0','c_i','lev','case'))
    Vwlz_per_c = ds.createVariable('wlz_per_c', 'f4', ('time', 'lat0','lon0','c_i','lev','case'))

    VUwind_mean = ds.createVariable('Uwind_mean', 'f4', ('time', 'lat0','lon0','lev1'))
    VVwind_mean = ds.createVariable('Vwind_mean', 'f4', ('time', 'lat0','lon0','lev1'))

    #write to file
    Vlat0[:] = latCenters
    Vlon0[:] = lonCenters
    Vlev0[:] = X.alt.values
    Vlev1[:] = zlev
    Vci[:] = c_central

    Vpower_per_c[:,:,:,:,:,:] = power_per_c[:,:,:,:,:,:]
    Vuw_per_c[:,:,:,:,:,:] = uw_per_c[:,:,:,:,:,:]
    Vuv_per_c[:,:,:,:,:,:] = uv_per_c[:,:,:,:,:,:]
    Vwl_per_c[:,:,:,:,:,:] = wl_per_c[:,:,:,:,:,:]
    Vwlz_per_c[:,:,:,:,:,:] = wlz_per_c[:,:,:,:,:,:]

    VUwind_mean[:,:,:,:] = Uwind_mean[:,:,:,:]
    VVwind_mean[:,:,:,:] = Vwind_mean[:,:,:,:]

    ds.close()